# Writing a workflow for the framework: specs and bindings

This notebook is for the authors of workflow packages, such as esssans.
A package that offers a workflow to the framework writes two things, both against `ess.spec`:

- a **spec**: the workflow's name, version, and two pydantic models: its params and its outputs. Apps build forms from it, the framework checks requests against it, and records name it. It is all that anyone outside the package sees.
- a **binding**: the code that computes the spec. Apart from the validators of the models and of the types in them, it is the only code of the package that the backend runs.

`ess.spec` needs only pydantic; a module that defines specs whose data are scipp objects also imports scipp.
`ess.spec.pipeline` adds sciline.
A workflow package depends on `ess.spec`, not on `ess.dispatch`, the client and backend that run it.

The thread: a reduction of one run, its spec and binding, then the same reduction over several runs, summed as they arrive by a *driver*.
At each step, what the framework gives the binding, and what the binding's author promises in return.

## 0. The workflow you have: a sciline pipeline

The toy SANS reduction of `toy_sans.py` stands in for esssans.
A run's file holds its counts per pixel and its monitor count; a sample run's file also holds the sample's transmission.
The detector is one row of four pixels, at x = 0, 1, 2, and 3 mm.
`load` stands for loading and correcting a run, the costly step.
A pixel's Q grows with its distance from the beam centre.
The reduction orders the pixels by that distance and sums them into Q bins (the numerator), divides by the monitor count times the transmission (the denominator), and subtracts the can's curve from the sample's.

The pipeline's keys for what a user gives are types that a spec can declare, defined in `toy_sans.py` as:

```python
SampleFile = NewType('SampleFile', Path)   # a sample run's file
CanFile = NewType('CanFile', Path)         # an empty-can run's file
IofQ = NewType('IofQ', sc.Variable)        # the result

class Reduction(BaseModel):
    bins: int = Field(default=2, gt=0)

class BeamCentre(BaseModel):               # in mm; (0, 0) is the nominal centre
    x: float = 0.0
    y: float = 0.0

class TransmissionFraction(BaseModel):
    value: float = Field(gt=0, le=1)

class Sample(BaseModel):                   # a sample run, and its transmission
    run: SampleFile
    transmission: TransmissionFraction | None = None  # None: the one in the file
```

A sample run and its transmission are one group, `Sample`, since a transmission belongs to its run: in a sum of runs, each run may have its own (section 4).
Providers read the run's file and its transmission from the group.

The reduction uses the beam centre as a scipp vector, `BeamCentrePosition`.
A provider makes it from the group:

```python
def beam_centre_position(centre: BeamCentre) -> BeamCentrePosition:
    return BeamCentrePosition(sc.vector([centre.x, centre.y, 0.0], unit='mm'))
```

So the pipeline's key is the group, which a spec can declare (section 1).

`toy_sans.write_runs` writes the toy runs as small files:

In [1]:
import tempfile
from pathlib import Path

import toy_sans
from toy_sans import (
    BeamCentre,
    CanFile,
    IofQ,
    Reduction,
    Sample,
    TransmissionFraction,
    make_pipeline,
)

folder = tempfile.TemporaryDirectory()
paths = toy_sans.write_runs(Path(folder.name))
print(paths[101].read_text())

workflow = make_pipeline({})
workflow[Sample] = Sample(run=paths[101])  # the transmission in the run's file
workflow[CanFile] = paths[110]
workflow[Reduction] = Reduction(bins=2)
workflow[BeamCentre] = BeamCentre()  # the nominal centre
workflow.compute(IofQ)

{"counts": [8.0, 6.0, 4.0, 2.0], "monitor": 10.0, "transmission": 0.8}


<scipp.Variable> (Q: 2)    float64  [dimensionless]  [1.55, 0.55]

## 1. The spec of one run

The spec declares two models:

| Model | Holds | In a request | When the workflow runs |
|---|---|---|---|
| params | everything the workflow takes: settings, files, data | each value inline or by reference | the values, and the data the references name |
| outputs | results | (none) | the data the workflow returns |

Each field of the params model is a *parameter*.
A request gives each value in one of two forms: *inline* or *by reference*.
An *inline value* is the value itself in the request; the record holds its JSON form.
A *reference* names data that exist elsewhere: a dataset, an output of a record, or an output of an accumulator.
The parameter's type decides which of the two it takes.
A type has a *JSON form* when it is not a file, holds no file, and pydantic makes a JSON schema of it with the params model's config:

| A parameter's type | Example | Inline | By reference |
|---|---|---|---|
| a file: `Path` or a `NewType` of it | `SampleFile` | no | to a dataset |
| has a JSON form | `Reduction`, `BeamCentre`, `TransmissionFraction` | yes | to an output that fits |
| a group that holds a file | `Sample` | field by field | to a dataset, for its one required file |
| anything else | `IofQ`, a scipp variable | no | to an output that fits |

A *group* is a pydantic model, such as `Reduction`: the usual type of a parameter given inline.
A group that holds a file, such as `Sample`, is given *field by field*: each of its fields inline or by reference, as a parameter is, so its file comes from a dataset.
A reference to a dataset stands for the whole group: its one required file, and its other fields at their defaults.
A *data value* is a value of a type without a JSON form: a file, or an object such as a scipp array.
A parameter's type is a type, possibly optional, or a list of one.
The settings, `reduction` and `beam_centre`, are in a base model of their own, `IofQSettings`, which the spec of several runs shares (section 4):

In [2]:
from pydantic import BaseModel, ConfigDict, Field

from ess.spec import WorkflowSpec


class IofQSettings(BaseModel):
    reduction: Reduction = Field(default_factory=Reduction)
    beam_centre: BeamCentre = Field(default_factory=BeamCentre)


class IofQParams(IofQSettings):
    sample: Sample
    can: CanFile


class IofQOutputs(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    iofq: IofQ


IOFQ = WorkflowSpec(
    name='toy-sans-iofq',
    version=1,
    title='I(Q)',
    description='I(Q) of one sample run, minus that of one can run.',
    params=IofQParams,
    outputs=IofQOutputs,
)

Rules of the spec, each with its reason:

- **The spec says what a value is, by its type, never where it lives.** A request names a dataset or another record's output; the binding gets the data read, a file as a local path. A file is a `NewType` of `Path`, and is given by reference to a dataset only, so that the dataset source checks access and each file a request reads has an identity in its record. A group holds a file only as one of its fields.
- **The form of a value belongs to the request, not to the spec.** One user types in a beam centre, and another computes it with a first reduction. `BeamCentre` is a group, so a request gives it inline, or by reference to an output of type `BeamCentre`, such as the `centre` that `FIND_CENTRE` in `toy_sans.py` outputs. The binding gets a `BeamCentre` either way. A value that the workflow uses as a scipp object, such as the beam centre's vector, is a group in the spec and converted by a provider, so that a user can type it in.
- **A type is a parameter's key in a sciline pipeline.** `SampleFile` and `CanFile` are both paths, so each is a `NewType`. `PipelineBinding` refuses two parameters with one key, unless `keys=` names another key for one.
- **Defaults live in the models only.** The binding gets every value filled in, so a default in the binding would never be used. A value *round-trips* when its JSON form reads back as the value. A default is a value, never a reference: of a type with a JSON form, a value that round-trips and whose JSON form is neither an `@ref` object nor a list holding one; of any other type, `None` or `[]`.
- **A default never changes which quantity an output is.** The default of `reduction` is a setting, and the transmission `None` of a `Sample` takes the one in the run's file: the output is I(Q) either way. The can run has no default. I(Q) without the can subtracted is another quantity, so it is another spec, and a request that leaves out the can is refused instead of computing the other quantity.
- **A check of one value belongs to its type.** `Reduction` refuses `bins=0`. Each parameter is checked alone, by its own field: a template, a stage call, or a push gives some values only ([README, Templates](../README.md#templates)). So `WorkflowSpec` checks this rule: The params model has no field validators, serializers, computed fields, or aliases, and no `default_factory` that takes the other values. A list parameter carries only length constraints, such as `Field(min_length=2)`. The params model may have model validators, but the framework never calls them. A group given field by field, such as `Sample`, follows the same rules, since a field of it may be given by reference.
- **A check across parameters is the binding's.** A binding may run the model validators, `IofQParams.model_validate(dict(params))`; the check then fails when the request runs.
- **Name and version identify the spec.** Records name them. `version` is the version of the interface: it changes with any change a consumer could observe, in the params or outputs models, or in what the workflow computes. The optional field `code_revision` names the code itself; records do not carry it.

An app reads the spec, not the package's code.
`serialize()` gives the schema of the params.
Each parameter takes the references it allows, each naming in `dataType` the type of data that fits, from which a picker of datasets or records is built; a parameter that takes an inline value also has its JSON Schema, from which a form is built:

In [3]:
serialized = IOFQ.serialize()
properties = serialized.params_schema['properties']
properties['beam_centre'], properties['sample']

({'anyOf': [{'$ref': '#/$defs/BeamCentre', 'not': {'required': ['@ref']}},
   {'anyOf': [{'$ref': '#/$defs/OutputRef'},
     {'$ref': '#/$defs/AccumulatorRef'}],
    'dataType': 'BeamCentre'}],
  'default': {'x': 0.0, 'y': 0.0},
  'title': 'Beam Centre'},
 {'anyOf': [{'$ref': '#/$defs/Sample', 'not': {'required': ['@ref']}},
   {'$ref': '#/$defs/DatasetRef', 'dataType': 'SampleFile'}],
  'title': 'Sample'})

A request lives in two phases.
In the *request phase*, before any data is read, a value given by reference is a reference, and the framework checks that the data it names fit the parameter's type.
An inline value is checked by its field.
In the *call phase*, the backend reads the data and builds the params instance from the values and the data, so every field holds the type it declares.
So the params model types the call phase only: it refuses what cannot run, and takes no reference:

In [4]:
from pydantic import ValidationError

from ess.spec import DatasetRef

try:
    IofQParams(reduction={'bins': 0}, sample={'run': paths[101]}, can=paths[110])
except ValidationError as error:
    print(error)
try:
    IofQParams(sample={'run': DatasetRef(dataset='run:101')}, can=paths[110])
except ValidationError as error:
    print(error)

1 validation error for IofQParams
reduction.bins
  Input should be greater than 0 [type=greater_than, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than
1 validation error for IofQParams
sample.run
  Input is not a valid path for <class 'pathlib.Path'> [type=path_type, input_value=DatasetRef(dataset='run:101'), input_type=DatasetRef]


## 2. The binding of one run

A binding is what the backend calls to compute a spec's requests.
The simplest binding is a function.
It takes an instance of the params model and returns one of the outputs model.
It gets every default filled in and every reference read:

In [5]:
from ess.spec import function_binding


def iofq(params: IofQParams) -> IofQOutputs:
    workflow = make_pipeline({})
    workflow[Reduction] = params.reduction
    workflow[BeamCentre] = params.beam_centre
    workflow[Sample] = params.sample
    workflow[CanFile] = params.can
    return IofQOutputs(iofq=workflow.compute(IofQ))


draft = function_binding(IOFQ, iofq)
params = IofQParams(sample=Sample(run=paths[101]), can=paths[110])
draft.compute(params)

IofQOutputs(iofq=<scipp.Variable> (Q: 2)    float64  [dimensionless]  [1.55, 0.55])

A request with every value given, a *plain request*, goes through `compute(params)`.
The spec is generic over its models, so a type checker checks `iofq` against `IOFQ`.

For a sciline pipeline, `PipelineBinding` does the same.
Each parameter sets the key of its type as written, here `Reduction`, `BeamCentre`, `Sample`, and `CanFile`, and each output is computed as the key of its type, here `IofQ`.
`make_pipeline` gets the fixed values by name, so that the graph may depend on them, and returns a pipeline of its own at each call:

In [6]:
from ess.spec.pipeline import PipelineBinding

single = PipelineBinding(IOFQ, make_pipeline)
single.compute(params)

IofQOutputs(iofq=<scipp.Variable> (Q: 2)    float64  [dimensionless]  [1.55, 0.55])

A pipeline with keys of its own names them with `keys=`, such as esssans with `keys={'can': Filename[BackgroundRun], 'iofq': BackgroundSubtractedIofQ}`.

A group is one key, so a provider takes it apart, as `read_sample` in `toy_sans.py` reads `sample.run`.

Each binding takes its spec, and the backend finds a request's binding by the spec's name and version.

**Promise: a binding modifies none of the values it gets, and returns no output that shares memory with one.**
The framework passes the values it holds without a copy, such as another record's output that a request references.
A binding that modified one, or returned a view of one, would change another record's value.
`forward` in `toy_sans.py` returns `params.curve['Q', 0].copy()` for this reason.

## 3. Stages: compute the fixed part once

When a user tunes `bins`, the runs stay the same.
The client then keeps a *stage*: a template whose fixed values stay, with *blanks*, the parameters left open (see [client-authors.ipynb](client-authors.ipynb)).
The backend calls `make_stage(fixed, blanks)` once, and the returned callable once per call through the stage, with the values of the blanks.
Both take values by name, each as its type, whether the request gave it inline or by reference: a group as the model, a file as a local path, any other data value as the object.
A parameter in neither, here `beam_centre`, takes its default.
`PipelineBinding` cuts the pipeline at the keys of the blanks with `sciline.Stage`, and computes what does not depend on them once:

In [7]:
fixed = {'sample': Sample(run=paths[101]), 'can': paths[110]}
call = single.make_stage(fixed, ('reduction',))
toy_sans.LOADED.clear()
for bins in (1, 2, 4):
    print(bins, call({'reduction': Reduction(bins=bins)}).iofq.values)
print('runs loaded:', len(toy_sans.LOADED))

1 [2.1]
2 [1.55 0.55]
4 [0.9  0.65 0.4  0.15]
runs loaded: 2


Two loads for three calls: the sample run and the can run, once each.
A function binding has no `make_stage`, since a stage over a function would hold nothing; the backend refuses a stage over it when the stage is made.

**Promise: a call through a stage gives the outputs of the plain request** (the *caching* symmetry).
The framework records a call through a stage as its plain request, so the record must not depend on how it was computed.
The `sciline.Stage` keeps this promise, given a pure pipeline.
`ess.spec.testing` checks it by computing both, and also checks that no output shares memory with a value the binding got:

In [8]:
from ess.spec.testing import check_caching

check_caching(single, fixed, [{'reduction': Reduction(bins=n)} for n in (2, 1, 2)])

## 4. The spec of several runs: tables

Many reductions combine runs: here, several sample runs and several can runs, summed.
A *table* is a list parameter whose *rows* an accumulator takes a push at a time, such as the sample runs.
A row that holds more than one value, such as a sample run and its transmission, is a group given field by field, `Sample`.
The multi-run spec has a params model of its own.
Each table is a list of the single-run type, here `list[Sample]` and `list[CanFile]`.
The other parameters, `reduction` and `beam_centre`, are those of `IofQSettings`, shared with `IofQParams`.
`tables` names the tables; a list parameter is a table only where `tables` names it:

In [9]:
class IofQMultiParams(IofQSettings):
    sample: list[Sample] = Field(min_length=1)
    can: list[CanFile] = Field(min_length=1)


IOFQ_MULTI = WorkflowSpec(
    name='toy-sans-iofq-multi',
    version=1,
    title='I(Q) of summed runs',
    description='I(Q) of the summed sample runs, minus that of the summed can runs.',
    params=IofQMultiParams,
    outputs=IofQOutputs,
    tables=('sample', 'can'),
)
IOFQ_MULTI.tables

('sample', 'can')

Rules of tables:

- **Each table is a list of the single-run type**, with at least one row, `Field(min_length=1)`. A table has no default, so a request gives every table, `[]` for one that may have no rows, and a template that forgets a table is refused; `WorkflowSpec` refuses a table with a default. The validators and constraints of the element type check each row: each `Sample` refuses a transmission above 1. The other parameters and the outputs model are those of `IOFQ`.
- **With one row in each table, the two specs agree**: a request of `IOFQ_MULTI` is then a request of `IOFQ`. `check_one_row` checks this (section 6).
- **Each row is given as its type allows.** Each field of a `Sample` is given inline or by reference, so one push may give the transmission of one run inline, and that of another by reference to an output of a record. A reference to a dataset stands for a `Sample` with the transmission in the run's file.
- **A row holds what belongs to one run.** A value that differs per run is in the run's file; a value that the file lacks or has wrong, such as a transmission, is a field of the row's group, so the request and its record hold it. Tables are independent of each other; which runs go together is the app's choice.
- **The rows of a table are distinct**: a row is in a table once. Rows are compared whole, as the record holds them, so the same run with another transmission is another row. For a table whose rows have a JSON form, such as time slices, the framework also compares the rows read, so a slice given inline and a reference to an equal output are one row.
- **A value given per run is not tuned alone.** A blank is a whole parameter, and a group is one key of the pipeline. A stage of `IOFQ` that tunes the transmission has `sample` as its blank, and loads the run again at each call.
- **Structure below a row is the binding's**, such as detector banks. The framework sees rows only.
- **A multi-run spec may have outputs of its own, or no single-run spec**, such as a stitch of all angles, or a sum of counts over the runs of a scan. It is written the same way: `angle: list[AngleFile] = Field(min_length=1)`, with `WorkflowSpec(..., tables=('angle',))`.

## 5. A driver: summing runs as they arrive

When runs arrive one at a time, a user wants the sum so far after each, without reducing the earlier runs again.
The client's *accumulator* takes one *push* at a time, each one or more rows of one table (see [client-authors.ipynb](client-authors.ipynb)).
Behind it is a *driver*: an object with `push(table, rows)` and `outputs()`, which the workflow package writes for one spec.
A push names its table and gives its rows, each as its type: a `Sample` with its run as a local path, or a can run's path.
Only a driver accumulates: the framework never decides how rows combine.

This driver sums each run's numerator and denominator per table, and computes I(Q) once from the sums.
It cuts the one-run pipeline with `sciline.Stage` twice: at the key of a table, `Sample` or `CanFile`, to compute one run's numerator and denominator, and at the four sums, to compute I(Q).
It gets the fixed values that are not tables, here the groups `reduction` and `beam_centre`, when it is made.
It takes the graph and the keys from the single-run binding, `single.pipeline(fixed)` and `single.keys`, so that it sets no key by hand:

In [10]:
from collections.abc import Mapping, Sequence
from typing import Any

import sciline
import scipp as sc

from toy_sans import CanRun, Denominator, Numerator, SampleRun


class IofQDriver:
    def __init__(self, fixed: Mapping[str, Any]) -> None:
        workflow = single.pipeline(fixed)
        self._contribute = {
            table: sciline.Stage(
                workflow,
                inputs=[single.keys[table]],
                outputs=(Numerator[run], Denominator[run]),
            )
            for table, run in (('sample', SampleRun), ('can', CanRun))
        }
        sums = [
            key[run] for key in (Numerator, Denominator) for run in (SampleRun, CanRun)
        ]
        self._finalize = sciline.Stage(workflow, inputs=sums, outputs=(IofQ,))
        self._sums: dict[str, dict[Any, sc.Variable]] = {}  # per table, by key

    def push(self, table: str, rows: Sequence[Any]) -> None:
        stage = self._contribute[table]
        for row in rows:
            parts = stage.compute({single.keys[table]: row})
            old = self._sums.get(table)
            self._sums[table] = (
                parts if old is None else {k: old[k] + v for k, v in parts.items()}
            )

    def outputs(self) -> IofQOutputs:
        if empty := sorted(self._contribute.keys() - self._sums.keys()):
            raise ValueError(f'no rows in {empty} yet')
        sums = {k: v for parts in self._sums.values() for k, v in parts.items()}
        return IofQOutputs(iofq=self._finalize.compute(sums)[IofQ])


groups = {'reduction': Reduction(bins=2), 'beam_centre': BeamCentre()}
driver = IofQDriver(groups)
toy_sans.LOADED.clear()
driver.push('sample', [Sample(run=paths[101])])
driver.push('can', [paths[110]])
print(driver.outputs().iofq.values)
driver.push(
    'sample',
    [
        Sample(run=paths[102]),
        Sample(run=paths[103], transmission=TransmissionFraction(value=0.6)),
    ],
)
print(driver.outputs().iofq.values)
print('runs loaded:', len(toy_sans.LOADED))

[1.55 0.55]
[1.2 0.4]
runs loaded: 4


Four loads for four runs: a push loads only its own rows.
Summing numerators and denominators apart, and dividing once, is what makes a sum of runs correct; a sum of normalized curves would not be.
The driver needs no spec, backend, or client, so a package uses and tests it as plain code.

The author of a driver promises, and the backend does not check:

- **`outputs()` gives the outputs of every row pushed so far**, however the rows were grouped into pushes, and whatever the order of pushes into different tables (the *arrival* and *order* symmetries). So each sum may depend on the fixed values and on one table only.
- **Within a table, the rows keep the order in which they were pushed.**
- **`push` does not modify a value it is given.** A value may be another record's output. A driver that adds in place copies the first contribution, so that adding never changes a value that another record returned. `IofQDriver` adds with `a + b`, which modifies nothing.
- **A driver may refuse a state**, such as one with no can row yet, by raising in `outputs()`. Later pushes may make the state valid.
- **A check across parameters or rows is the driver's.** A driver gets each value checked by its own field only; the framework runs no model validator of the params model.

In return, `outputs()` may share memory with what the driver holds: the backend is done with the outputs of a state before it pushes the next rows.

`check_arrival_and_order` takes the function that makes a driver, the fixed values that are not tables, and the rows of each table by its name.
It pushes the rows in several schedules, one row at a time and in groups, table by table and alternating, and reads between pushes.
It compares each state with what a new driver gives after one push per table, skipping states that the driver refuses.
It also fails if a push modifies the values it was given:

In [11]:
from ess.spec.testing import check_arrival_and_order

tables = {
    'sample': [
        Sample(run=paths[101]),
        Sample(run=paths[102]),
        Sample(run=paths[103], transmission=TransmissionFraction(value=0.6)),
    ],
    'can': [paths[110], paths[111]],
}
check_arrival_and_order(IofQDriver, groups, tables)

Both sides of the comparison go through the driver, so the check does not prove the outputs right.

A sum that depends on two tables would break the order symmetry: the can's sum would change with the sample rows pushed before it.
esssans has this case, since its pixel masks come from the sample run.
Its driver must give outputs that do not depend on the order of pushes, for example by keeping the can rows that a mask applies to; `check_arrival_and_order` catches a driver whose outputs do.

A workflow that needs every row at once, such as a fit over all runs, writes a driver that keeps its rows and computes over all of them in `outputs()`.
The outputs are the same as for a driver that sums; only the cost differs, and which to pay is the workflow's choice.

## 6. The binding of several runs

`DriverBinding` binds the multi-run spec to the function that makes a driver.
A plain request makes a driver, pushes the rows of each table in one push, and reads the outputs:

In [12]:
from ess.spec import DriverBinding

multi = DriverBinding(IOFQ_MULTI, IofQDriver)
multi.compute(IOFQ_MULTI.params(**tables))

IofQOutputs(iofq=<scipp.Variable> (Q: 2)    float64  [dimensionless]  [1.2, 0.4])

For an accumulator, the backend calls `make_accumulator(fixed, blanks)`, with the tables that the pushes fill as the blanks.
The binding makes a driver from the fixed values that are not tables, and pushes the rows of each table in `fixed`, such as a can run fixed by the client's template.
The backend then pushes each client push into the driver:

In [13]:
driver = multi.make_accumulator(
    {'reduction': Reduction(bins=2), 'can': [paths[110]]}, ('sample',)
)
driver.push('sample', [Sample(run=paths[101])])
driver.outputs()

IofQOutputs(iofq=<scipp.Variable> (Q: 2)    float64  [dimensionless]  [1.55, 0.55])

**Promise: with one row in each table, the binding of the multi-run spec gives the outputs of the single-run spec** (the *one-row* symmetry).
The specs agree: with one row in each table, a request of `IOFQ_MULTI` is a request of `IOFQ`, since `IofQMultiParams` shares the settings of `IofQParams` and each of its tables is a list of the single-run type.
`one_row` makes that request from a request of `IOFQ`:

In [14]:
from ess.spec.testing import check_one_row, one_row

params = IofQParams(
    beam_centre=BeamCentre(x=1.25),
    sample=Sample(run=paths[103], transmission=TransmissionFraction(value=0.6)),
    can=paths[110],
)
one_row(IOFQ_MULTI, params)

IofQMultiParams(reduction=Reduction(bins=2), beam_centre=BeamCentre(x=1.25, y=0.0), sample=[Sample(run=PosixPath('/tmp/tmptyyzhtxt/run-103.json'), transmission=TransmissionFraction(value=0.6))], can=[PosixPath('/tmp/tmptyyzhtxt/run-110.json')])

The package keeps the promise: the driver must compute what the pipeline computes.
`check_one_row` computes both and compares them; it is the only check against an independent reference:

In [15]:
check_one_row(single, multi, params)

Each binding serves some of the client's actions, and the backend refuses the others when a stage or an accumulator is made:

| Binding | Plain request | Stage | Accumulator |
|---|---|---|---|
| `PipelineBinding` | `compute` | `make_stage` | refused |
| `DriverBinding` | `compute` | refused | `make_accumulator` |
| `function_binding` | `compute` | refused | refused |

## 7. What the framework gives the binding

The binding may rely on this:

| Call | When, and with what |
|---|---|
| `compute(params)` | once per plain request; an instance of the params model, with data read and defaults filled in, each value checked by its own field and no model validator run; possibly from several threads at once |
| `make_stage(fixed, blanks)` | once per stage of a client, at its first call; every value but the blanks, by name; possibly from several threads at once |
| the callable it returns | once per call through the stage, with the values of the blanks, by name; possibly from several threads at once |
| `make_accumulator(fixed, blanks)` | once per accumulator; every value but the tables that the pushes fill, by name. `DriverBinding` calls its function that makes a driver here, and in `compute`, possibly from several threads at once |
| `driver.push(table, rows)` | once per client push, with its rows, data read; in the order logged; one thread at a time; never while the readers of the state before it are busy |
| `driver.outputs()` | once per state that someone reads; one thread at a time |

Each value the binding gets holds the type its field declares, whether the request gave it inline or by reference: the backend checks each inline value by its field and each reference's fit by type when a request is made, and each value by its field when it is read.
[framework-promises.ipynb](framework-promises.ipynb) traces these calls for one accumulator.

## Summary: what the author of a workflow promises

| Promise | Checked by |
|---|---|
| each parameter's type is a type, possibly optional, or a list of one; a group holds a file only as a field, and is then given field by field | `WorkflowSpec` |
| a value that users give both inline and by reference is a group, converted by a provider where the workflow uses a scipp object | nobody |
| each parameter has a key of its own in the pipeline | `PipelineBinding` |
| defaults live in the models only, and are values | `WorkflowSpec` checks that a default is a value; nobody checks the rest |
| a check of one value belongs to its type; a check across parameters is the binding's | `WorkflowSpec` checks the rule on the params model in section 1; nobody checks the rest |
| a new version when the params or outputs models change | nobody |
| the binding returns no output that shares memory with a value it gets | `check_one_row`, `check_caching` (numpy-backed values) |
| the binding modifies no value it gets | nobody, except for the values a push gets (`check_arrival_and_order`) |
| the pipeline is pure, so a call through a stage gives the outputs of the plain request | `check_caching` |
| the multi-run spec with one row per table takes the requests of the single-run spec, and gives its outputs | `check_one_row` |
| after each push, the driver's outputs are those of all rows pushed so far, in any grouping and in any order across tables | `check_arrival_and_order` |
| `push` modifies no value | `check_arrival_and_order` |
| structure below a row belongs to the binding | nobody |

A package registers its bindings through the entry-point group `ess.dispatch.workflows`.
The entry-point group is decided and not implemented; a notebook or app passes bindings to `local(..., bind=toy_sans.BINDINGS)`.

In [16]:
folder.cleanup()